# cuda-empty-cache — ex1: periodic cache release in an eval loop

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `cuda-empty-cache`. Running the final beacon cell reports progress against the `PyTorch: torch.cuda.empty_cache` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: torch.cuda.empty_cache` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cuda-empty-cache`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cuda-empty-cache"
DD_SUBTOPIC = "PyTorch: torch.cuda.empty_cache"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.cuda.empty_cache — what it actually does

PyTorch's CUDA allocator keeps a **block cache** of GPU memory that was allocated and then freed by Python but NOT returned to the driver. Future allocations reuse cached blocks (fast). `nvidia-smi` reports the cached memory as 'used' even though no live tensor occupies it.

`torch.cuda.empty_cache()` releases ALL cached blocks back to the CUDA driver. After the call, `nvidia-smi` drops — but no live tensor is affected. The call does NOT free live tensors. It does NOT speed up training. It does NOT reduce peak memory. Its ONLY purpose is to make GPU memory visible to OTHER processes (or to `nvidia-smi`).

**Mocking for CPU.** This drill is CPU-friendly: we replace `torch.cuda.empty_cache` with a mock so the function runs everywhere and we can assert it was called the expected number of times.

### Exercise 1 — periodic cache release in an eval loop

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `torch.cuda.empty_cache()` at a controlled cadence inside an eval loop, verifying via mock that the call count matches the intended cadence and that no live tensor is destroyed.
> Keywords: cuda, memory, cache, mock
> ```

**KCs targeted:** `empty-cache-semantics`, `empty-cache-no-live-tensor-impact`

Implement `ex1_eval_loop_with_cache_release(batches, release_every)`. The standard 'free cached blocks every K batches' eval pattern:

1. `batches` is a list of tensors (each is one eval batch).
2. For each batch: compute a per-batch sum (`b.sum()`) and append the scalar to a running list of results.
3. After every `release_every` batches (1-indexed: so if `release_every=3`, call after batch 3, 6, 9, …) call `torch.cuda.empty_cache()`.
4. Return the list of per-batch sums (as a 1-D float tensor).

The test patches `torch.cuda.empty_cache` with a mock so it works on CPU and counts call frequency.

Input: `batches: list[Tensor]`, `release_every: int >= 1`.
Output: 1-D float tensor of length `len(batches)`.

In [ ]:
def ex1_eval_loop_with_cache_release(batches, release_every: int) -> Tensor:
    sums = []
    for i, b in enumerate(batches, start=1):
        sums.append(b.sum())
        if i % release_every == 0:
            t.cuda.empty_cache()
    return t.stack(sums)


<details><summary>Solution</summary>

```python
def ex1_eval_loop_with_cache_release(batches, release_every: int) -> Tensor:
    sums = []
    for i, b in enumerate(batches, start=1):
        sums.append(b.sum())
        if i % release_every == 0:
            t.cuda.empty_cache()
    return t.stack(sums)
```

**`empty_cache` does NOT free live tensors.** The test's 'live-tensor invariant' check is the whole point — even after the cache is released, every batch in the input list is still a valid tensor with all its data.

**When to actually call it.** Three legitimate reasons: (1) handing the GPU back to another process; (2) before measuring memory with `nvidia-smi`; (3) before a known-spiky allocation that needs a fresh contiguous block. NEVER as a 'magic fix' for OOM — if you are OOM the live tensors are too big, not the cache.

**Why mock for testing.** The function is pure side-effect on a global allocator. Mocking lets us verify call cadence without needing real CUDA hardware in CI.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()